<a href="https://colab.research.google.com/github/shravan1808/ML_SERIES/blob/Main/15_Gradient_Boosting_Early_Stopping/notebook/Project_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

In [3]:
def fetch_network_logs(url):
  try:
    response = requests.get(url)
    response.raise_for_status()
    data = pd.read_csv(io.StringIO(response.text))
    return data
  except requests.exceptions.RequestException as e:
    print(f"Error fetching data: {e}")

In [4]:
DATASET_API_ENDPOINT = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

In [5]:
df = fetch_network_logs(DATASET_API_ENDPOINT)

In [6]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [7]:
df.shape

(344, 7)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [9]:
df.dropna(inplace=True)

In [10]:
df['Outage_Risk'] = np.where(df['species']=='Gentoo',1,0)

In [15]:
X= df.drop(['species','Outage_Risk','island','sex'],axis=1)
y= df['Outage_Risk']

In [20]:
y_counts = y.value_counts()

In [16]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.3,stratify=y)

In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
X_train_scaled_shape = X_train_scaled.shape
X_test_scaled_shape = X_test_scaled.shape

In [30]:
model = GradientBoostingClassifier(n_estimators=500,learning_rate=0.05,validation_fraction=0.15,n_iter_no_change=10,tol=1e-4,random_state=42)
model.fit(X_train_scaled,y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]

print(model.n_estimators_)
print(model.train_score_[-1])

118
0.002475383183748232


In [32]:
test_accuracy = accuracy_score(y_test,y_pred)
test_log_loss = log_loss(y_test,y_prob)
test_roc_auc = roc_auc_score(y_test,y_prob)
print(f"Test Accuracy: {test_accuracy*100}%")
print(f"Test Log Loss: {test_log_loss:.4f}")
print(f"Test ROC AUC: {test_roc_auc:.4f}")

Test Accuracy: 99.0%
Test Log Loss: 0.0794
Test ROC AUC: 0.9857


In [34]:
print("\n========== GRADIENT BOOSTING & EARLY STOPPING OPTIMIZATION ==========")

print(f"\n{'Data Ingestion Status':<28}: REST API Ingestion Successful (HTTP 200 OK)")
print(f"{'Master Dataset Records':<28}: {len(df)} Telemetry Records")
print(f"{'Model Architecture':<28}: GradientBoostingClassifier (Early Stopping Enabled)")

print("\nEarly Stopping Audit:")
print(f"- {'Maximum Tree Boundary':<24}: {model.n_estimators} Trees")
print(f"- {'Optimal Stopped Trees':<24}: {model.n_estimators_} Trees")

compute_savings = (
    (model.n_estimators - model.n_estimators_) /
    model.n_estimators
) * 100

print(f"- {'Compute Savings':<24}: {compute_savings:.1f}% reduction in boosting iterations")

print("\nPerformance Metrics:")
print(f"- {'Test Accuracy':<24}: {test_accuracy * 100:.2f}%")
print(f"- {'Test Log Loss':<24}: {test_log_loss:.4f}")
print(f"- {'Test ROC-AUC':<24}: {test_roc_auc:.4f}")

print("\nConclusion:")
print(
    "Monitoring out-of-fold validation loss during gradient boosting "
    "prevents over-parameterization, stopping training as soon as "
    "test generalization peaks."
)


========== GRADIENT BOOSTING & EARLY STOPPING OPTIMIZATION ==========

Data Ingestion Status       : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records      : 333 Telemetry Records
Model Architecture          : GradientBoostingClassifier (Early Stopping Enabled)

Early Stopping Audit:
- Maximum Tree Boundary   : 500 Trees
- Optimal Stopped Trees   : 118 Trees
- Compute Savings         : 76.4% reduction in boosting iterations

Performance Metrics:
- Test Accuracy           : 99.00%
- Test Log Loss           : 0.0794
- Test ROC-AUC            : 0.9857

Conclusion:
Monitoring out-of-fold validation loss during gradient boosting prevents over-parameterization, stopping training as soon as test generalization peaks.
